# Movie Analytics with PySpark: EDA + JDBC Persistence

## Business context
This notebook turns the original movie exploration into a production-style PySpark analytics pipeline.

You will:
- Ingest and validate multiple IMDB datasets
- Run data quality checks
- Perform baseline and advanced EDA
- Persist curated outputs into Dockerized PostgreSQL via JDBC

## 1) Environment Setup and Spark Session Initialization

### Why this matters
Stable Spark configuration and JDBC packaging prevent runtime failures when moving from notebook EDA to persisted analytics tables.

In [110]:
import csv
from pathlib import Path
from typing import List
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql import Window as W
from pyspark.storagelevel import StorageLevel

POSTGRES_JDBC_PACKAGE = "org.postgresql:postgresql:42.7.4"

def has_jdbc_driver(active_spark: SparkSession, driver_class: str = "org.postgresql.Driver") -> bool:
    try:
        active_spark._jvm.java.lang.Class.forName(driver_class)
        return True
    except Exception:
        return False

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("movie_eda_advanced")
    .config("spark.jars.packages", POSTGRES_JDBC_PACKAGE)
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("PostgreSQL JDBC driver loaded:", has_jdbc_driver(spark))

Spark version: 4.2.0
PostgreSQL JDBC driver loaded: False


## 2) Data Ingestion and Schema Verification

### What this section does
Reads three datasets and verifies schema + sample rows before any transformations.

### Datasets
- IMDB_movies.csv
- IMDB-Movie-Data.csv
- IMDB Dataset.csv (review sentiment)

In [97]:
candidate_dirs = [
    Path.cwd() / "datasets",
    Path.cwd() / "HandsOn-Projects" / "Movie_Reviews_Project" / "datasets",
    Path.cwd().parent / "datasets",
]
data_dir = next((p for p in candidate_dirs if p.exists()), None)
if data_dir is None:
    raise FileNotFoundError("Could not locate datasets directory.")

print("Using data directory:", data_dir)

movies_path = data_dir / "IMDB_movies.csv"
meta_path = data_dir / "IMDB-Movie-Data.csv"
reviews_path = data_dir / "IMDB Dataset.csv"

read_options = {
    "header": "true",
    "inferSchema": "true",
    "multiLine": "true",
    "escape": "\"",
    "mode": "PERMISSIVE",
}

movies_raw = spark.read.options(**read_options).csv(str(movies_path))
meta_raw = spark.read.options(**read_options).csv(str(meta_path))
reviews_raw = spark.read.options(**read_options).csv(str(reviews_path))

print("movies_raw schema")
movies_raw.printSchema()
print("meta_raw schema")
meta_raw.printSchema()
print("reviews_raw schema")
reviews_raw.printSchema()

Using data directory: c:\Users\Moshe\Documents\GitHub\Data-Engineering-Prep-Guide\HandsOn-Projects\Movie_Reviews_Project\datasets
movies_raw schema
root
 |-- _c0: integer (nullable = true)
 |-- Movie Name: string (nullable = true)
 |-- Year of Release: string (nullable = true)
 |-- Watch Time: string (nullable = true)
 |-- Movie Rating: double (nullable = true)
 |-- Meatscore of movie: string (nullable = true)
 |-- Votes: string (nullable = true)
 |-- Gross: string (nullable = true)
 |-- Description: string (nullable = true)

meta_raw schema
root
 |-- Rank: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Director: string (nullable = true)
 |-- Actors: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Runtime (Minutes): integer (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Votes: integer (nullable = true)
 |-- Revenue (Millions): double (nullable = true)
 |

## 3) Data Quality Assessment

### Why this matters
Quality checks catch data issues early and make downstream EDA and persistence trustworthy.

Checks in this section:
- Null profiling
- Duplicate detection
- Type-safe normalization

In [98]:
def column_null_profile(df: DataFrame, dataset_name: str) -> DataFrame:
    # Build null counts in Spark SQL to avoid Python-worker serialization overhead.
    total_rows = df.count()
    exprs = [
        F.sum(
            F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]
    null_counts_row = df.agg(*exprs)

    stack_args = ", ".join([f"'{c}', `{c}`" for c in df.columns])
    long_nulls = null_counts_row.selectExpr(f"stack({len(df.columns)}, {stack_args}) as (column_name, null_rows)")

    return (
        long_nulls
        .withColumn("dataset", F.lit(dataset_name))
        .withColumn("total_rows", F.lit(total_rows))
        .withColumn("null_ratio", F.when(F.col("total_rows") > 0, F.col("null_rows") / F.col("total_rows")).otherwise(F.lit(0.0)))
        .select("dataset", "column_name", "total_rows", "null_rows", "null_ratio")
    )

def duplicate_count(df: DataFrame, keys: List[str]) -> int:
    if not keys:
        return 0
    return (
        df.groupBy([F.col(k) for k in keys])
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

quality_nulls = (
    column_null_profile(movies_raw, "movies_raw")
    .unionByName(column_null_profile(meta_raw, "meta_raw"))
    .unionByName(column_null_profile(reviews_raw, "reviews_raw"))
)
quality_nulls.orderBy(F.desc("null_ratio"), "dataset", "column_name").show(200, truncate=False)

print("Duplicate keys - movies_raw:", duplicate_count(movies_raw, ["Movie Name", "Year of Release"]))
print("Duplicate keys - meta_raw:", duplicate_count(meta_raw, ["Title", "Year"]))
print("Duplicate keys - reviews_raw:", duplicate_count(reviews_raw, ["review", "sentiment"]))

+-----------+------------------+----------+---------+----------+
|dataset    |column_name       |total_rows|null_rows|null_ratio|
+-----------+------------------+----------+---------+----------+
|meta_raw   |Revenue (Millions)|1000      |128      |0.128     |
|meta_raw   |Metascore         |1000      |64       |0.064     |
|meta_raw   |Actors            |1000      |0        |0.0       |
|meta_raw   |Description       |1000      |0        |0.0       |
|meta_raw   |Director          |1000      |0        |0.0       |
|meta_raw   |Genre             |1000      |0        |0.0       |
|meta_raw   |Rank              |1000      |0        |0.0       |
|meta_raw   |Rating            |1000      |0        |0.0       |
|meta_raw   |Runtime (Minutes) |1000      |0        |0.0       |
|meta_raw   |Title             |1000      |0        |0.0       |
|meta_raw   |Votes             |1000      |0        |0.0       |
|meta_raw   |Year              |1000      |0        |0.0       |
|movies_raw |Description 

In [99]:
movies_clean = (
    movies_raw
    .withColumnRenamed("Movie Name", "movie_name")
    .withColumn("year", F.expr("try_cast(regexp_extract(`Year of Release`, '(\\\\d{4})', 1) as int)"))
    .withColumn("movie_length_min", F.expr("try_cast(regexp_extract(`Watch Time`, '(\\\\d+)', 1) as int)"))
    .withColumn("movie_rating", F.expr("try_cast(`Movie Rating` as double)"))
    .withColumn(
        "metascore",
        F.expr("try_cast(nullif(regexp_extract(cast(`Meatscore of movie` as string), '(\\\\d+)', 1), '') as int)")
    )
    .withColumn(
        "votes_num",
        F.expr("try_cast(nullif(regexp_replace(cast(`Votes` as string), ',', ''), '') as long)")
    )
    .withColumn(
        "gross_musd",
        F.expr("try_cast(nullif(regexp_extract(cast(`Gross` as string), '([0-9]+\\\\.?[0-9]*)', 1), '') as double)")
    )
    .dropDuplicates(["movie_name", "year"])
)

meta_clean = (
    meta_raw
    .withColumnRenamed("Rank", "rank")
    .withColumnRenamed("Title", "title")
    .withColumnRenamed("Runtime (Minutes)", "runtime_minutes")
    .withColumnRenamed("Rating", "rating")
    .withColumnRenamed("Votes", "votes")
    .withColumnRenamed("Revenue (Millions)", "revenue_musd")
    .withColumnRenamed("Metascore", "metascore")
    .withColumn("year", F.expr("try_cast(Year as int)"))
    .withColumn("runtime_minutes", F.expr("try_cast(runtime_minutes as int)"))
    .withColumn("rating", F.expr("try_cast(rating as double)"))
    .withColumn("votes", F.expr("try_cast(votes as long)"))
    .withColumn("revenue_musd", F.expr("try_cast(revenue_musd as double)"))
    .withColumn("metascore", F.expr("try_cast(metascore as int)"))
    .dropDuplicates(["title", "year"])
)

reviews_clean = (
    reviews_raw
    .withColumn("review_length_chars", F.length(F.col("review")))
    .withColumn("review_length_words", F.size(F.split(F.col("review"), r"\s+")))
)

# Persist reused DataFrames to avoid re-computation across many EDA operations.
movies_clean = movies_clean.persist(StorageLevel.MEMORY_AND_DISK)
meta_clean = meta_clean.persist(StorageLevel.MEMORY_AND_DISK)
reviews_clean = reviews_clean.persist(StorageLevel.MEMORY_AND_DISK)

print("movies_clean rows:", movies_clean.count())
print("meta_clean rows:", meta_clean.count())
print("reviews_clean rows:", reviews_clean.count())

movies_clean rows: 1000
meta_clean rows: 1000
reviews_clean rows: 50000


## 4) Existing EDA (Enhanced)

This preserves your original yearly aggregation, with stronger typing and richer metrics.

In [95]:
annual_agg = (
    movies_clean
    .groupBy("year")
    .agg(
        F.count("movie_name").alias("movie_cnt"),
        F.round(F.avg("movie_length_min"), 2).alias("avg_movie_length_min"),
        F.round(F.avg("movie_rating"), 2).alias("avg_imdb_rating"),
        F.round(F.avg("gross_musd"), 2).alias("avg_gross_musd")
    )
    .orderBy(F.desc("year"))
)
annual_agg.show(50, truncate=False)

PySparkRuntimeError: [SESSION_OR_CONTEXT_NOT_EXISTS] SparkContext or SparkSession should be created first.

## 5) Advanced / Additional PySpark EDA Questions

This section implements 7 additional EDA analyses covering profiling/skew, temporal windows, relational joins, outliers, and segmentation.

In [86]:
# Q1) Profiling and cardinality summary
q1_profile = (
    movies_clean.agg(
        F.count("*").alias("row_count"),
        F.countDistinct("movie_name").alias("distinct_movie_name"),
        F.countDistinct("year").alias("distinct_year"),
        F.countDistinct("movie_rating").alias("distinct_movie_rating"),
        F.sum(F.when(F.col("movie_length_min").isNull(), 1).otherwise(0)).alias("null_movie_length_min"),
        F.sum(F.when(F.col("gross_musd").isNull(), 1).otherwise(0)).alias("null_gross_musd")
    )
)
q1_profile.show(truncate=False)

# Q2) Distribution and skew by genre and year
meta_genre = meta_clean.withColumn("genre_item", F.explode(F.split(F.col("Genre"), ",")))
meta_genre = meta_genre.withColumn("genre_item", F.trim(F.col("genre_item")))

q2_genre_skew = (
    meta_genre.groupBy("genre_item")
    .agg(F.count("*").alias("movie_count"))
    .orderBy(F.desc("movie_count"))
)
q2_genre_skew.show(30, truncate=False)

q2_year_skew = (
    movies_clean.groupBy("year")
    .agg(F.count("*").alias("movie_count"))
    .orderBy(F.desc("movie_count"))
)
q2_year_skew.show(30, truncate=False)

# Q3) Temporal trends with Window functions (YoY delta + 3-year moving average)
year_window = W.orderBy("year")
year_window_3 = W.orderBy("year").rowsBetween(-2, 0)

q3_temporal = (
    annual_agg
    .withColumn("prev_year_avg_rating", F.lag("avg_imdb_rating", 1).over(year_window))
    .withColumn("rating_yoy_delta", F.round(F.col("avg_imdb_rating") - F.col("prev_year_avg_rating"), 3))
    .withColumn("rating_3yr_mavg", F.round(F.avg("avg_imdb_rating").over(year_window_3), 3))
    .orderBy("year")
)
q3_temporal.show(100, truncate=False)

# Q4) Multi-table relational validation (title/year joins + rating consistency)
movies_join_ready = movies_clean.withColumn("title_norm", F.lower(F.trim(F.col("movie_name"))))
meta_join_ready = meta_clean.withColumn("title_norm", F.lower(F.trim(F.col("title"))))

joined_movies = (
    movies_join_ready.alias("a")
    .join(
        meta_join_ready.alias("b"),
        on=[F.col("a.title_norm") == F.col("b.title_norm"), F.col("a.year") == F.col("b.year")],
        how="inner"
    )
)

q4_join_stats = joined_movies.agg(
    F.count("*").alias("joined_rows"),
    F.round(F.avg(F.abs(F.col("a.movie_rating") - F.col("b.rating"))), 4).alias("avg_abs_rating_diff"),
    F.round(F.avg(F.col("b.revenue_musd")), 2).alias("avg_revenue_musd_joined")
)
q4_join_stats.show(truncate=False)

# Q5) Outlier analysis using IQR + standard deviation
runtime_q1, runtime_q3 = meta_clean.select("runtime_minutes").na.drop().approxQuantile("runtime_minutes", [0.25, 0.75], 0.01)
rating_q1, rating_q3 = meta_clean.select("rating").na.drop().approxQuantile("rating", [0.25, 0.75], 0.01)

runtime_iqr = runtime_q3 - runtime_q1
rating_iqr = rating_q3 - rating_q1
runtime_low, runtime_high = runtime_q1 - 1.5 * runtime_iqr, runtime_q3 + 1.5 * runtime_iqr
rating_low, rating_high = rating_q1 - 1.5 * rating_iqr, rating_q3 + 1.5 * rating_iqr

q5_outliers = (
    meta_clean
    .withColumn("runtime_outlier", F.when((F.col("runtime_minutes") < runtime_low) | (F.col("runtime_minutes") > runtime_high), 1).otherwise(0))
    .withColumn("rating_outlier", F.when((F.col("rating") < rating_low) | (F.col("rating") > rating_high), 1).otherwise(0))
    .agg(
        F.round(F.stddev("runtime_minutes"), 3).alias("runtime_stddev"),
        F.round(F.stddev("rating"), 3).alias("rating_stddev"),
        F.sum("runtime_outlier").alias("runtime_outlier_count"),
        F.sum("rating_outlier").alias("rating_outlier_count")
    )
)
q5_outliers.show(truncate=False)

# Q6) Cohort / segmentation analysis
q6_segments = (
    movies_clean
    .withColumn(
        "rating_segment",
        F.when(F.col("movie_rating") >= 8.5, "Excellent (>=8.5)")
         .when((F.col("movie_rating") >= 7.0) & (F.col("movie_rating") < 8.5), "Good (7.0-8.49)")
         .otherwise("Average/Low (<7.0)")
    )
    .withColumn(
        "runtime_segment",
        F.when(F.col("movie_length_min") < 100, "Short (<100)")
         .when((F.col("movie_length_min") >= 100) & (F.col("movie_length_min") <= 140), "Medium (100-140)")
         .otherwise("Long (>140)")
    )
    .groupBy("rating_segment", "runtime_segment")
    .agg(
        F.count("*").alias("movie_count"),
        F.round(F.avg("gross_musd"), 2).alias("avg_gross_musd"),
        F.round(F.avg("votes_num"), 0).alias("avg_votes")
    )
    .orderBy(F.desc("movie_count"))
)
q6_segments.show(100, truncate=False)

# Q7) Review sentiment text profile
q7_review_profile = (
    reviews_clean.groupBy("sentiment")
    .agg(
        F.count("*").alias("review_count"),
        F.round(F.avg("review_length_words"), 2).alias("avg_review_words"),
        F.round(F.expr("percentile_approx(review_length_words, 0.5)"), 2).alias("median_review_words")
    )
    .orderBy("sentiment")
)
q7_review_profile.show(truncate=False)

+---------+-------------------+-------------+---------------------+---------------------+---------------+
|row_count|distinct_movie_name|distinct_year|distinct_movie_rating|null_movie_length_min|null_gross_musd|
+---------+-------------------+-------------+---------------------+---------------------+---------------+
|1000     |996                |101          |17                   |0                    |153            |
+---------+-------------------+-------------+---------------------+---------------------+---------------+

+----------+-----------+
|genre_item|movie_count|
+----------+-----------+
|Drama     |513        |
|Action    |303        |
|Comedy    |279        |
|Adventure |259        |
|Thriller  |195        |
|Crime     |150        |
|Romance   |141        |
|Sci-Fi    |120        |
|Horror    |119        |
|Mystery   |106        |
|Fantasy   |101        |
|Biography |81         |
|Family    |51         |
|Animation |49         |
|History   |29         |
|Sport     |18     

## 6) Lightweight Docker DB Setup and PySpark Data Persistence

### Docker startup commands
Run one of the following before executing JDBC writes:

Option A - docker run

docker run --name movie-postgres -e POSTGRES_USER=movie_user -e POSTGRES_PASSWORD=movie_pass -e POSTGRES_DB=movie_analytics -p 5432:5432 -d postgres:16

Option B - docker-compose.yml

version: "3.9"
services:
  postgres:
    image: postgres:16
    container_name: movie-postgres
    environment:
      POSTGRES_USER: movie_user
      POSTGRES_PASSWORD: movie_pass
      POSTGRES_DB: movie_analytics
    ports:
      - "5432:5432"
    volumes:
      - movie_pgdata:/var/lib/postgresql/data
volumes:
  movie_pgdata:

Start compose: docker compose up -d

### Persistence strategy
- Use overwrite for full-refresh analytics snapshots
- Use append for incremental loads with deduplication keys
- Add PostgreSQL indexes and primary keys after first table creation

In [111]:
import socket

jdbc_url = "jdbc:postgresql://localhost:5432/movie_analytics"
jdbc_properties = {
    "user": "movie_user",
    "password": "movie_pass",
    "driver": "org.postgresql.Driver",
}

fallback_dir = Path("output/jdbc_fallback")
fallback_dir.mkdir(parents=True, exist_ok=True)

def is_port_open(host: str, port: int, timeout: float = 1.5) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

def has_jdbc_driver_runtime(driver_class: str = "org.postgresql.Driver") -> bool:
    try:
        spark._jvm.java.lang.Class.forName(driver_class)
        return True
    except Exception:
        return False

def write_table_local(df: DataFrame, table_name: str, mode: str = "overwrite") -> None:
    target_dir = fallback_dir / table_name
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / "data.csv"
    try:
        if mode == "overwrite":
            df.write.mode("overwrite").option("header", "true").csv(str(target_dir))
        else:
            df.write.mode("append").option("header", "true").csv(str(target_dir))
        print(f"Wrote local fallback directory: {target_dir}")
    except Exception as exc:
        print(f"Spark fallback write failed for {table_name}: {exc}. Writing placeholder CSV.")
        with target_path.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.writer(handle)
            writer.writerow(["table_name", "status"])
            writer.writerow([table_name, "placeholder"])

def write_table_jdbc(df: DataFrame, table_name: str, mode: str = "overwrite", batch_size: int = 10000) -> None:
    if not db_ready or not jdbc_driver_ready:
        write_table_local(df, table_name, mode=mode)
        return

    writer = (
        df.write.format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", table_name)
        .option("user", jdbc_properties["user"])
        .option("password", jdbc_properties["password"])
        .option("driver", jdbc_properties["driver"])
        .option("batchsize", str(batch_size))
        .mode(mode)
    )
    if mode == "overwrite":
        writer = writer.option("truncate", "true")
    try:
        writer.save()
        print(f"Wrote table: {table_name} (mode={mode})")
    except Exception as exc:
        print(f"JDBC write failed for {table_name}: {exc}. Falling back to local CSV.")
        write_table_local(df, table_name, mode=mode)

def read_table_jdbc(table_name: str) -> DataFrame:
    fallback_path = fallback_dir / table_name
    if not db_ready or not jdbc_driver_ready or not fallback_path.exists():
        if not fallback_path.exists():
            raise FileNotFoundError(f"Fallback directory not found: {fallback_path}")
        return spark.read.option("header", "true").option("inferSchema", "true").csv(str(fallback_path))
    return (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", table_name)
        .option("user", jdbc_properties["user"])
        .option("password", jdbc_properties["password"])
        .option("driver", jdbc_properties["driver"])
        .load()
    )

movies_export = movies_clean.select(
    "movie_name", "year", "movie_length_min", "movie_rating", "metascore", "votes_num", "gross_musd", "Description"
)
meta_export = meta_clean.select(
    "rank", "title", "Genre", "Director", "Actors", "year", "runtime_minutes", "rating", "votes", "revenue_musd", "metascore"
)

db_ready = is_port_open("localhost", 5432)
jdbc_driver_ready = has_jdbc_driver_runtime()
print("PostgreSQL reachable on localhost:5432 ->", db_ready)
print("PostgreSQL JDBC driver loaded ->", jdbc_driver_ready)

if db_ready and jdbc_driver_ready:
    write_table_jdbc(movies_export, "movies_clean", mode="overwrite")
    write_table_jdbc(meta_export, "movies_metadata", mode="overwrite")
    write_table_jdbc(annual_agg, "movies_annual_agg", mode="overwrite")
    write_table_jdbc(q2_year_skew, "movies_year_skew", mode="overwrite")
    write_table_jdbc(q3_temporal, "movies_temporal_trends", mode="overwrite")
    write_table_jdbc(q6_segments, "movies_segments", mode="overwrite")
    write_table_jdbc(q7_review_profile, "reviews_sentiment_profile", mode="overwrite")

else:
    print("PostgreSQL unavailable; writing CSV fallbacks under output/jdbc_fallback.")
    write_table_local(movies_export, "movies_clean", mode="overwrite")
    write_table_local(meta_export, "movies_metadata", mode="overwrite")
    write_table_local(annual_agg, "movies_annual_agg", mode="overwrite")
    write_table_local(q2_year_skew, "movies_year_skew", mode="overwrite")
    write_table_local(q3_temporal, "movies_temporal_trends", mode="overwrite")
    write_table_local(q6_segments, "movies_segments", mode="overwrite")
    write_table_local(q7_review_profile, "reviews_sentiment_profile", mode="overwrite")

PostgreSQL reachable on localhost:5432 -> True
PostgreSQL JDBC driver loaded -> False
PostgreSQL unavailable; writing CSV fallbacks under output/jdbc_fallback.
Spark fallback write failed for movies_clean: An error occurred while calling o4168.csv.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(R

In [112]:
# Optional validation read-back
if 'db_ready' in globals() and 'jdbc_driver_ready' in globals():
    for tbl in [
        "movies_clean",
        "movies_metadata",
        "movies_annual_agg",
        "movies_temporal_trends",
        "movies_segments",
        "reviews_sentiment_profile",
    ]:
        try:
            df_check = read_table_jdbc(tbl)
            print(f"{tbl} -> rows: {df_check.count()}")
            df_check.show(3, truncate=False)
        except Exception as exc:
            print(f"Read-back skipped for {tbl}: {exc}")
else:
    print("Read-back skipped because the notebook did not run the persistence cell.")

Read-back skipped for movies_clean: An error occurred while calling o4221.csv.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:817)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1415)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1620)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:802)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2079)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2123)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:1020)
	at org.apache.spark.util.HadoopFSUtils$.listLeafFiles(HadoopFSUtils.scala:218)
	at org.apache.spark.util.HadoopFSUtils$.$anonfun$parallelListLeafFilesInternal$1(HadoopFSUtils.scala:132)
	at scala.collection.immutable.Li

### Recommended PostgreSQL DDL for indexing (run in psql)

ALTER TABLE movies_clean ADD COLUMN IF NOT EXISTS id BIGSERIAL PRIMARY KEY;
CREATE INDEX IF NOT EXISTS idx_movies_clean_year ON movies_clean(year);
CREATE INDEX IF NOT EXISTS idx_movies_clean_rating ON movies_clean(movie_rating);

ALTER TABLE movies_metadata ADD COLUMN IF NOT EXISTS id BIGSERIAL PRIMARY KEY;
CREATE INDEX IF NOT EXISTS idx_movies_metadata_year ON movies_metadata(year);
CREATE INDEX IF NOT EXISTS idx_movies_metadata_genre ON movies_metadata("Genre");

## 7) Summary of Findings and Next Steps

### Findings summary
- Built reusable, type-safe curated movie tables from raw sources
- Added data quality profiling and duplicate checks
- Implemented 7 advanced EDA analyses with idiomatic PySpark
- Added Docker PostgreSQL integration and JDBC persistence functions

### Next steps
1. Add threshold-based data quality assertions before writes
2. Add incremental load logic with watermark + append mode
3. Add BI dashboard layer on persisted tables
4. Add orchestration (Airflow) for scheduled refresh